# Exploratory notebook

For viewing the data in different phases of the pipeline.

## 0. Raw landing data

In [0]:
# List files in landing volume
display(dbutils.fs.ls("/Volumes/workouts_demo/bronze/landing"))

## 1. Bronze data 

In [0]:
catalog = "workouts_demo"
bronze_tbl = f"{catalog}.bronze.activities_raw"

spark.sql(f"USE CATALOG {catalog}")
spark.sql("USE SCHEMA bronze")
display(spark.sql(f"SELECT COUNT(*) AS rows FROM {bronze_tbl}"))

In [0]:
display(spark.sql(f"SELECT activity_id, activity_type, distance17, elapsed_time5, moving_time FROM {bronze_tbl} LIMIT 8"))

## 2. Silver

### Table checks

In [0]:
silver_tbl = f"{catalog}.silver.activities"
spark.sql("USE SCHEMA silver")
display(spark.sql(f"SELECT COUNT(*) AS rows FROM {silver_tbl}"))

In [0]:
display(spark.sql(f"SELECT * FROM {silver_tbl} LIMIT 8"))

### Exploratory plots

In [0]:

import seaborn as sns
import matplotlib.pyplot as plt
sdf = spark.table(silver_tbl)
pdf = sdf.limit(100_000).toPandas()


In [0]:
# Plotting time vs. distance
pdf["distance_km"] = pdf["distance_m"]/1000.0
pdf["time_h"] = pdf["moving_time_s"]/3600.0
pdf["elev_per_dist"] = pdf["elevation_m"]/pdf["distance_km"]
sns.scatterplot(data=pdf, y="distance_km", x="time_h", hue="sport", alpha=0.7)
plt.title("Time vs. distance by sport")

plt.xlabel("Moving time (hours)")
plt.ylabel("Distance (km)")
plt.show()

In [0]:
# Plotting time vs. proportional elevation gain
sns.barplot(data=pdf, y="elev_per_dist", x="sport")
plt.title("Elevation gain by sport")

plt.xlabel("Sport")
plt.ylabel("Elevation gain per distance (m/km)")
plt.show()

In [0]:
# Plotting time vs. elevation gain
sns.barplot(data=pdf, y="distance_km", x="sport")
plt.title("Distance by sport")

plt.xlabel("Sport")
plt.ylabel("Distance (km)")
plt.show()

In [0]:
# Plotting time vs. elevation gain
sns.barplot(data=pdf, y="distance_km", x="activity_month_int")
plt.title("Distance by month")

plt.xlabel("Sport")
plt.ylabel("Distance (km)")
plt.show()

## 3. Gold

In [0]:
gold_tbl = f"{catalog}.gold.summary_by_sport"
spark.sql("USE SCHEMA gold")
display(spark.sql(f"SELECT * FROM {gold_tbl}"))

In [0]:
gold_tbl_2 = f"{catalog}.gold.longest_activities"
spark.sql("USE SCHEMA gold")
display(spark.sql(f"SELECT * FROM {gold_tbl_2} SORT BY distance_km DESC LIMIT 15"))

In [0]:
gold_tbl_3 = f"{catalog}.gold.summary_by_month"
spark.sql("USE SCHEMA gold")
display(spark.sql(f"SELECT * FROM {gold_tbl_3}"))

In [0]:
sdf = spark.table(gold_tbl_3)
pdf = sdf.limit(100_000).toPandas()

In [0]:
barplot = sns.barplot(data=pdf, y="activities", x="activity_month_int", hue="sport")
plt.title("Activites by month")
plt.ylabel("Number of activities")
plt.xlabel("Month")
plt.show()